# **Cross-Lingual Phonetic Transliteration (English → Italian)**

This notebook implements the full experimental pipeline described in the accompanying paper: dataset construction from the CMU Pronouncing Dictionary, subword/character tokenizer training, and the training and evaluation of two sequence-to-sequence architectures — a BLSTM with Bahdanau Attention (PyTorch, trained from scratch) and a compact BART (Hugging Face Transformers, trained from scratch), each combined with two source-side tokenization strategies (BPE, Unigram).

**Repository:** [gaiacitro/Cross-Lingual-Phonetic-Translitteration](https://github.com/gaiacitro/Cross-Lingual-Phonetic-Translitteration)

## Environment Setup and Repository Cloning

Sets up the working environment: the project repository and the external dependencies (`transformers`, `sentencepiece`, `jiwer`) not natively available on Colab.

In [ ]:
!git clone https://github.com/gaiacitro/Cross-Lingual-Phonetic-Translitteration.git
%cd Cross-Lingual-Phonetic-Translitteration
!git pull
!ls

!pip install transformers sentencepiece jiwer

## Data Preprocessing and Dataset Generation

Constructs the parallel English–Italian dataset used for training. `cmudict.dict` (the CMU Pronouncing Dictionary with test-set entries commented out to prevent leakage) is parsed and converted into Italian phonetic transliterations through the deterministic mapping in `mapping_cmu_italian.py`.

In [ ]:
# JSONL dataset and TXT files generation
!python dataset_creation.py

## Training of BPE, Unigram, and Char tokenizers

Trains the SentencePiece tokenizers used by the models: BPE and Unigram (vocabulary size 300) for the English source side, and a character-level tokenizer for the Italian target side.

In [ ]:
# Training BPE, Unigram, and Character-level tokenizers
!python tokenization.py

## Training Phase


Each configuration below (architecture × tokenizer) is trained independently, with a maximum of 10 epochs and early stopping (patience 3) based on validation CER.

### 1. BART Model Training: BPE Tokenization Strategy

In [ ]:
# Initiating BART architecture training protocol
!python train_bart_bpe.py

# Compressing the model directory into a .zip archive
!zip -r bart_bpe_best_model.zip ./bart_bpe_best_model

# Invoking Colab interface for local file download
from google.colab import files
files.download('bart_bpe_best_model.zip')

### 2. BART Model Training: Unigram Tokenization Strategy

In [ ]:
# Initiating BART architecture training protocol with Unigram strategy
!python train_bart_unigram.py

# Compressing the model directory into a .zip archive
!zip -r bart_unigram_best_model.zip ./bart_unigram_best_model

# Invoking Colab interface for local file download
from google.colab import files
files.download('bart_unigram_best_model.zip')

### 3. BLSTM Model Training with Attention: BPE Tokenization Strategy

In [ ]:
!python train_blstm_att_bpe.py

from google.colab import files
files.download('blstm_att_bpe_best.pth')

### 4. BLSTM Model Training with Attention: Unigram Tokenization Strategy

In [ ]:
!python train_blstm_att_unigram.py

from google.colab import files
files.download('blstm_att_unigram_best.pth')

## Model Weights Loading via Google Drive

Retrieves previously trained checkpoints from Google Drive, avoiding the need to repeat training when only inference or model inspection is required.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive to the Colab environment
drive.mount('/content/drive')

# 2. Define source (Drive) and destination (Local Colab) paths
drive_models_path = "/content/drive/MyDrive/NLP/best_models"
local_models_path = "/content/Cross-Lingual-Phonetic-Translitteration/best_models"

if os.path.exists(drive_models_path):
    print("Folder 'best_models' found on Drive! Copying files...")

    # Create the local directory and copy all contents (.pth and .zip)
    !mkdir -p {local_models_path}
    !cp -r {drive_models_path}/* {local_models_path}/

    # Navigate to the local models folder to extract the BART archives
    print("Extracting ZIP archives...")
    %cd {local_models_path}

    # Unzip all .zip files silently (-q) and overwrite existing files (-o)
    !unzip -q -o "*.zip"

    # Return to the root project directory
    %cd /content/Cross-Lingual-Phonetic-Translitteration

    print("✅ Models successfully loaded and extracted! Ready for inference.")
else:
    print("❌ WARNING: 'best_models' folder not found on Google Drive under 'NLP/best_models'. Please check the path.")

## Inference for Test Phase

Evaluation of the selected model on the held-out test set, used to compute the final Character Error Rate.

In [ ]:
from inference import run_inference

#run_inference("blstm_att_bpe")
run_inference("blstm_att_unigram")
#run_inference("bart_bpe")
#run_inference("bart_unigram")


## Authors

- **Gaia Citro** — `citro.2026094@studenti.uniroma1.it`
- **Lucia Fornetti** — `fornetti.2214370@studenti.uniroma1.it`

Sapienza University of Rome — NLP 2024–2025

